In [1]:
import os
import shutil
from google.colab import drive

# /content/drive varsa ve mount değilse temizle
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive")

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

os.makedirs(f"{project_path}/data/raw", exist_ok=True)
os.makedirs(f"{project_path}/data/processed", exist_ok=True)
os.makedirs(f"{project_path}/outputs/metrics", exist_ok=True)
os.makedirs(f"{project_path}/outputs/samples", exist_ok=True)

print("Project path:", project_path)
print("Exists:", os.path.exists(project_path))

Project path: /content/drive/MyDrive/turkish_legal_rag
Exists: True


In [3]:
!pip install datasets

In [4]:
from datasets import load_dataset

dataset = load_dataset("Renicames/turkish-law-chatbot")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/13354 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})


In [6]:
import pandas as pd

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

In [7]:
train_df.head()
train_df.columns

Index(['Soru', 'Cevap'], dtype='object')

In [8]:
# Kolon isimlerini standart yapalım
train_df = train_df.rename(columns={"Soru": "question", "Cevap": "answer"})
test_df = test_df.rename(columns={"Soru": "question", "Cevap": "answer"})

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (13354, 2)
Test shape: (1500, 2)


,question,answer
0,"Anayasa madde 1'e göre, türkiye'nin devlet şek...","Anayasa madde 1'e göre, türkiye'nin devlet şek..."
1,Anayasa madde 1'de belirtilen cumhuriyetin tan...,"Anayasa madde 1'de belirtilen cumhuriyet, halk..."
2,"Anayasa madde 1, cumhuriyetin ilan edilmesini ...","Anayasa madde 1, cumhuriyetin ilan edilmesini,..."
3,"Anayasa madde 1, cumhuriyetin hangi tarihte il...","Anayasa madde 1, türkiye cumhuriyeti'nin 29 ek..."
4,"Anayasa madde 1'e göre, cumhuriyetin temel öze...","Anayasa madde 1'e göre, cumhuriyetin temel öze..."


In [9]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())

print("Train duplicates:", train_df.duplicated().sum())
print("Test duplicates:", test_df.duplicated().sum())

question    0
answer      0
dtype: int64
question    0
answer      0
dtype: int64
Train duplicates: 0
Test duplicates: 0


In [10]:
from sklearn.model_selection import train_test_split

# HF train setinden validation ayırıyoruz
train_qa_df, val_qa_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

# HF test zaten final test olarak kullanılacak
test_qa_df = test_df.copy()

print("Train QA:", train_qa_df.shape)
print("Validation QA:", val_qa_df.shape)
print("Test QA:", test_qa_df.shape)

Train QA: (11350, 2)
Validation QA: (2004, 2)
Test QA: (1500, 2)


In [11]:
train_qa_df.to_csv(f"{project_path}/data/processed/train_qa.csv", index=False, encoding="utf-8-sig")
val_qa_df.to_csv(f"{project_path}/data/processed/val_qa.csv", index=False, encoding="utf-8-sig")
test_qa_df.to_csv(f"{project_path}/data/processed/test_qa.csv", index=False, encoding="utf-8-sig")

print("QA datasets saved.")

QA datasets saved.


In [12]:
print("Saved files:")
print(f"{project_path}/data/processed/train_qa.csv")
print(f"{project_path}/data/processed/val_qa.csv")
print(f"{project_path}/data/processed/test_qa.csv")

Saved files:
/content/drive/MyDrive/turkish_legal_rag/data/processed/train_qa.csv
/content/drive/MyDrive/turkish_legal_rag/data/processed/val_qa.csv
/content/drive/MyDrive/turkish_legal_rag/data/processed/test_qa.csv


In [13]:
kaggle_path = f"{project_path}/data/raw/turkish_law_dataset.csv"

print("Kaggle file exists:", os.path.exists(kaggle_path))

Kaggle file exists: True


In [14]:
import pandas as pd

kaggle_df = pd.read_csv(kaggle_path)

print("Shape:", kaggle_df.shape)
print("Columns:", kaggle_df.columns.tolist())

kaggle_df.head()

Shape: (13954, 6)
Columns: ['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'Score']


,soru,cevap,veri türü,kaynak,context,Score
0,"Anayasa, Türk Vatanı ve Milletinin ebedi varlı...","Anayasa, Türk Vatanı ve Milletinin ebedi varlı...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
1,"Anayasa, Türkiye Cumhuriyetinin hangi milliyet...","Anayasa, Türkiye Cumhuriyetinin kurucusu olan ...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
2,"Anayasa, Türkiye Cumhuriyetini hangi konumda t...","Anayasa, Türkiye Cumhuriyetini dünya milletler...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
3,"Anayasa, Türkiye Cumhuriyetinin hangi hedefler...","Anayasa, Türkiye Cumhuriyetinin ebedi varlığın...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
4,"Anayasa, egemenliğin kime ait olduğunu nasıl b...","Anayasa, egemenliğin kayıtsız şartsız Türk Mil...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,10


In [15]:
kaggle_clean = kaggle_df.copy()
kaggle_clean.columns = [c.strip().lower() for c in kaggle_clean.columns]

required_cols = ["soru", "cevap", "context"]

kaggle_clean = kaggle_clean.dropna(subset=required_cols)
kaggle_clean = kaggle_clean.drop_duplicates()

for col in required_cols:
    kaggle_clean[col] = kaggle_clean[col].astype(str).str.strip()

kaggle_clean = kaggle_clean[
    (kaggle_clean["soru"].str.len() > 10) &
    (kaggle_clean["cevap"].str.len() > 10) &
    (kaggle_clean["context"].str.len() > 50)
].reset_index(drop=True)

kaggle_clean = kaggle_clean.drop_duplicates(subset=["soru"]).reset_index(drop=True)

print("Clean Kaggle shape:", kaggle_clean.shape)
kaggle_clean.head()

Clean Kaggle shape: (12885, 6)


,soru,cevap,veri türü,kaynak,context,score
0,"Anayasa, Türk Vatanı ve Milletinin ebedi varlı...","Anayasa, Türk Vatanı ve Milletinin ebedi varlı...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
1,"Anayasa, Türkiye Cumhuriyetinin hangi milliyet...","Anayasa, Türkiye Cumhuriyetinin kurucusu olan ...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
2,"Anayasa, Türkiye Cumhuriyetini hangi konumda t...","Anayasa, Türkiye Cumhuriyetini dünya milletler...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
3,"Anayasa, Türkiye Cumhuriyetinin hangi hedefler...","Anayasa, Türkiye Cumhuriyetinin ebedi varlığın...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
4,"Anayasa, egemenliğin kime ait olduğunu nasıl b...","Anayasa, egemenliğin kayıtsız şartsız Türk Mil...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,10


In [16]:
from sklearn.model_selection import train_test_split

kaggle_train, kaggle_temp = train_test_split(
    kaggle_clean,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

kaggle_val, kaggle_test = train_test_split(
    kaggle_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Kaggle train:", kaggle_train.shape)
print("Kaggle val:", kaggle_val.shape)
print("Kaggle test:", kaggle_test.shape)

Kaggle train: (9019, 6)
Kaggle val: (1933, 6)
Kaggle test: (1933, 6)


In [17]:
kaggle_train.to_csv(f"{project_path}/data/processed/kaggle_train_qa.csv", index=False, encoding="utf-8-sig")
kaggle_val.to_csv(f"{project_path}/data/processed/kaggle_val_qa.csv", index=False, encoding="utf-8-sig")
kaggle_test.to_csv(f"{project_path}/data/processed/kaggle_test_qa.csv", index=False, encoding="utf-8-sig")

print("Kaggle QA splits saved.")

Kaggle QA splits saved.


In [18]:
retrieval_source_df = kaggle_clean[["context", "kaynak"]].copy()
retrieval_source_df = retrieval_source_df.drop_duplicates(subset=["context"]).reset_index(drop=True)

print("Unique context count:", retrieval_source_df.shape)
retrieval_source_df.head()

Unique context count: (240, 2)


,context,kaynak
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,Türkiye Cumhuriyeti Anayasası
1,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,Türkiye Cumhuriyeti Anayasası
2,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,Türkiye Cumhuriyeti Anayasası
3,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,Türkiye Cumhuriyeti Anayasası
4,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...,Türkiye Cumhuriyeti Anayasası


In [47]:
import re

def normalize_legal_text(text):
    text = str(text)
    text = text.replace("\\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_by_legal_structure(text):
    text = normalize_legal_text(text)

    # Madde geçen her yeri ayırır: satır başı şartı yok
    text = re.sub(
        r"(?i)(Madde\s+\d+\s*[–-])",
        r"\n\n\1",
        text
    )

    # A. Genel olarak / B. Kanunların... gibi başlıkları ayırır
    text = re.sub(
        r"([A-ZÇĞİÖŞÜ]\.\s+[A-ZÇĞİÖŞÜa-zçğıöşü])",
        r"\n\n\1",
        text
    )

    parts = re.split(r"\n\s*\n+", text)
    return [p.strip() for p in parts if p.strip()]


def split_long_block_by_sentence(block, max_len=800):
    block = block.strip()

    if len(block) <= max_len:
        return [block]

    sentences = re.split(r"(?<=[.!?;:])\s+", block)

    chunks = []
    current = ""

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if len(current) + len(sent) + 1 <= max_len:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent

    if current:
        chunks.append(current)

    return chunks


def merge_short_chunks(chunks, min_len=120, max_len=900):
    merged = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        # Madde başlıyorsa öncekiyle birleştirme
        if re.search(r"(?i)^Madde\s+\d+\s*[–-]", chunk):
            merged.append(chunk)
        elif merged and len(chunk) < min_len and len(merged[-1]) + len(chunk) + 1 <= max_len:
            merged[-1] = merged[-1] + " " + chunk
        else:
            merged.append(chunk)

    return merged


def build_legal_chunks(text, min_len=120, max_len=800):
    blocks = split_by_legal_structure(text)

    chunks = []
    for block in blocks:
        chunks.extend(split_long_block_by_sentence(block, max_len=max_len))

    chunks = merge_short_chunks(chunks, min_len=min_len, max_len=max_len + 100)
    chunks = [c.strip() for c in chunks if len(c.strip()) >= 50]

    return chunks

In [48]:
all_chunks = []

for idx, row in retrieval_source_df.iterrows():
    chunks = build_legal_chunks(row["context"], min_len=120, max_len=800)

    for chunk in chunks:
        all_chunks.append({
            "source_context_id": f"kaggle_ctx_{idx:05d}",
            "source": row["kaynak"],
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)
chunks_df = chunks_df.drop_duplicates(subset=["chunk_text"]).reset_index(drop=True)

chunks_df["chunk_id"] = ["chunk_" + str(i).zfill(6) for i in range(len(chunks_df))]
chunks_df = chunks_df[["chunk_id", "source_context_id", "source", "chunk_text"]]

chunks_df["chunk_len"] = chunks_df["chunk_text"].str.len()

print("Retrieval chunks:", chunks_df.shape)
print(chunks_df["chunk_len"].describe())

Retrieval chunks: (3775, 5)
count    3775.000000
mean      370.538278
std       206.229327
min        50.000000
25%       202.000000
50%       321.000000
75%       512.000000
max      1205.000000
Name: chunk_len, dtype: float64


In [49]:
for i in range(7, 15):
    print("=" * 100)
    print("Chunk ID:", chunks_df.loc[i, "chunk_id"])
    print("Length:", chunks_df.loc[i, "chunk_len"])
    print(chunks_df.loc[i, "chunk_text"])

Chunk ID: chunk_000007
Length: 142
ÜÇÜNCÜ KISIM CUMHURİYETİN TEMEL ORGANLARI BİRİNCİ BÖLÜM Yasama Öncesi… I I. Türkiye Büyük Millet Meclisinin görev ve yetkileri A. Genel olarak
Chunk ID: chunk_000008
Length: 518
Türkiye Büyük Millet Meclisinin görev ve yetkileri, kanun koymak, değiştirmek ve kaldırmak; bütçe ve kesinhesap kanun tekliflerini görüşmek ve kabul etmek; para basılmasına ve savaş ilânına karar vermek; milletlerarası andlaşmaların onaylanmasını uygun bulmak, Türkiye Büyük Millet Meclisi üye tamsayısının beşte üç çoğunluğunun kararı ile genel ve özel af ilânına karar vermek ve Anayasanın diğer maddelerinde öngörülen yetkileri kullanmak ve görevleri yerine getirmektir. B. Kanunların teklif edilmesi ve görüşülmesi
Chunk ID: chunk_000009
Length: 219
Madde 88 – Kanun teklif etmeye (…)[40] milletvekilleri yetkilidir. Kanun (…)[41] tekliflerinin Türkiye Büyük Millet Meclisinde görüşülme usul ve esasları içtüzükle düzenlenir. C. Kanunların Cumhurbaşkanınca yayımlanması
Chunk ID: chu

In [50]:
chunks_df.to_csv(
    f"{project_path}/data/processed/retrieval_corpus.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Retrieval corpus saved.")

Retrieval corpus saved.


In [51]:
import os

for filename in os.listdir(f"{project_path}/data/processed"):
    print(filename)

train_qa.csv
val_qa.csv
test_qa.csv
kaggle_train_qa.csv
kaggle_val_qa.csv
kaggle_test_qa.csv
retrieval_corpus.csv
